# G3-VPR v3 — Notebook Chấm Điểm Cross-Domain & Đối Chiếu SOTA Trực Tiếp Trên Kaggle
**Không cần train lại!** Notebook này tự động:
1. Tìm toàn bộ **10 checkpoint `G3-VPR v3`** bạn đã train (từ Output của notebook train trước hoặc Dataset bạn gắn vào Input).
2. Tự động tải về các bộ test khác miền chuẩn CVPR (**`SPED`**, **`AmsterTime`**, và tự nhận thêm **`Pitts30k` / `MSLS-val` / `Tokyo24/7` / `Nordland`** nếu bạn gắn vào Input).
3. Chấm ở chế độ **`BS=1` tuyệt đối** cho: **`G3-VPR v3` (`128..4096-D + G-OTR + AM-VPR Conformal Cascade`)**, **9 dòng Ablation đồng ngân sách**, và **3 đối thủ SOTA (`MegaLoc`, `DINOv2-SALAD`, `EigenPlaces`)**.
4. Xuất sẵn bảng báo cáo **`CVPR_V3_SUMMARY_REPORT.md`** + nén vào **`g3vpr_v3_crossdomain_eval.zip`** để tải về đối chiếu ngay!

**Cách chạy nhanh nhất trên Kaggle:**
1. **Settings** bên phải: chọn **Accelerator → `GPU T4`** (hoặc `T4 x2`), bật **Internet → `ON`**.
2. **Input** bên phải: bấm **`+ Add Input` → tab `Your Work` → chọn Output của Notebook train `G3-VPR v3` vừa chạy xong** (hoặc upload `g3vpr_paper_artifacts_v3.zip` làm Dataset).
3. Bấm **`Save Version` → `Save & Run All`** (~10–20 phút là xong toàn bộ)!

In [ ]:
# Cell 1: Giai nen code G3-VPR v3 & cai thu vien cham diem (khong can train lai!)
import base64, io, os, subprocess, sys, tarfile, torch

WORK_ROOT = "/kaggle/working/D2VPR_eval" if os.path.isdir("/kaggle/working") else os.path.abspath("./D2VPR_eval")
os.makedirs(WORK_ROOT, exist_ok=True)

ARCHIVE_B64 = (
    "..."
    "ruIqruIqruIqruIqruL6V1z/B08UzMsA4AYA"
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE_B64)), mode="r:gz") as tf:
    tf.extractall(WORK_ROOT)
os.chdir(WORK_ROOT)
print("[OK] Da giai nen code G3-VPR v3 tai:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (hay bat GPU T4!)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "gdown"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pytorch-metric-learning"], check=True)
print("[OK] San sang cham diem!")

In [ ]:
# Cell 2: Tu dong tim toan bo 10 checkpoint G3-VPR v3 trong /kaggle/input (hoac giai nen tu file zip neu co)
import os, zipfile
if os.path.isdir("/kaggle/working/D2VPR_eval"):
    os.chdir("/kaggle/working/D2VPR_eval")

CKPT_ROOT = None
M5_CKPT = None

# 1) Quet truc tiep /kaggle/input tim g3vpr_universal_best.pth / g3vpr_best_fp16.pth
if os.path.isdir("/kaggle/input"):
    for r, dirs, files in os.walk("/kaggle/input"):
        if "g3vpr_M5_full" in dirs:
            cand = os.path.join(r, "g3vpr_M5_full", "g3vpr_universal_best.pth")
            cand16 = os.path.join(r, "g3vpr_M5_full", "g3vpr_best_fp16.pth")
            if os.path.isfile(cand):
                CKPT_ROOT, M5_CKPT = r, cand
                break
            elif os.path.isfile(cand16):
                CKPT_ROOT, M5_CKPT = r, cand16
                break
        for fn in files:
            if fn.endswith(".zip") and "g3vpr_paper_artifacts_v3" in fn:
                print("Tim thay file zip artifacts, dang giai nen:", os.path.join(r, fn))
                with zipfile.ZipFile(os.path.join(r, fn), "r") as zf:
                    zf.extractall("./v3_extracted")
                CKPT_ROOT = "./v3_extracted/checkpoints"
                M5_CKPT = os.path.join(CKPT_ROOT, "g3vpr_M5_full", "g3vpr_universal_best.pth")
                break
        if M5_CKPT:
            break

assert M5_CKPT and os.path.isfile(M5_CKPT), (
    "Khong tim thay checkpoint G3-VPR v3 trong /kaggle/input! "
    "Hay bam + Add Input -> Your Work -> gan Output cua notebook G3-VPR v3 (hoac dataset chua g3vpr_paper_artifacts_v3.zip)."
)
print("[OK] Main checkpoint M5_full:", M5_CKPT)
print("[OK] Thu muc chua cac ablation:", CKPT_ROOT)
print("Cac thu muc checkpoint tim thay:", sorted(os.listdir(CKPT_ROOT)))

In [ ]:
# Cell 3: Tu dong tai cac bo test khac mien chuan CVPR (SPED, AmsterTime, SVOX, ...) neu chua gan trong /kaggle/input
import glob, os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR_eval"):
    os.chdir("/kaggle/working/D2VPR_eval")

DATASETS_DIR = os.path.abspath("./datasets")
os.makedirs(DATASETS_DIR, exist_ok=True)

# Clone cong cu tai chuan VPR-datasets-downloader & cai du dependencies (py3_wget, utm, ...)
dl_repo = "./VPR-datasets-downloader"
if not os.path.isdir(dl_repo):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gmberton/VPR-datasets-downloader.git", dl_repo], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py3_wget", "utm", "pillow", "tqdm"], check=True)
avail_scripts = {os.path.basename(p): p for p in glob.glob(f"{dl_repo}/download_*.py")}
print("Cac script tai co san:", sorted(avail_scripts.keys()))

# Tai cac bo benchmark chuan CVPR (SPED, AmsterTime, SVOX, Tokyo24/7)
for ds_name in ["sped", "amstertime", "svox", "tokyo247"]:
    target_dir = os.path.join(DATASETS_DIR, ds_name, "images", "test")
    if os.path.isdir(target_dir):
        print(f"[Da co san] {ds_name}")
        continue
    sc_path = avail_scripts.get(f"download_{ds_name}.py")
    if not sc_path:
        print(f"[Bo qua] Khong co script cho {ds_name}")
        continue
    print(f"\n=== Dang tai bo benchmark chuan: {ds_name} ===")
    ret = subprocess.run([sys.executable, sc_path, "--datasets_folder", DATASETS_DIR], check=False)
    if ret.returncode != 0:
        print(f"[Canh bao] Khong tai tu dong duoc {ds_name}, tiep tuc bo khac.")

print("\n[OK] Danh sach thu muc trong ./datasets:", sorted(os.listdir(DATASETS_DIR)))

In [ ]:
# Cell 4: Chay Cham Diem Toan Dien G3-VPR v3 + G-OTR + Conformal Cascade + 9 Ablations + SOTA Hub (BS=1) + FAISS 1M
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR_eval"):
    os.chdir("/kaggle/working/D2VPR_eval")

eval_cmd = [
    sys.executable, "-u", "eval_cvpr_v3_suite.py",
    "--ckpt", M5_CKPT,
    "--checkpoints_dir", CKPT_ROOT,
    "--datasets_root", "./datasets",
    "--query_bs", "1",
    "--db_bs", "64",
    "--run_sota_hub",
    "--run_faiss_1m",
    "--output_dir", "./results_crossdomain",
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

In [ ]:
# Cell 5: In toan bo bao cao doi chieu & Dong goi ZIP ket qua de tai ve may!
import os, shutil
if os.path.isdir("/kaggle/working/D2VPR_eval"):
    os.chdir("/kaggle/working/D2VPR_eval")

report_md = "./results_crossdomain/CVPR_V3_SUMMARY_REPORT.md"
if os.path.isfile(report_md):
    print("=" * 85)
    print(open(report_md, encoding="utf-8").read())
    print("=" * 85)

zip_out = "/kaggle/working/g3vpr_v3_crossdomain_eval" if os.path.isdir("/kaggle/working") else "./g3vpr_v3_crossdomain_eval"
archive = shutil.make_archive(zip_out, "zip", "./results_crossdomain")
print("\n[HOAN TAT] Tai file ket qua tai muc Output ben phai:", archive)